# Identificación y autoría
**Speaker:** Eduardo Osorio Venegas  
**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI).  
Material educativo con apoyo generativo en las celdas Markdown; requiere revisión y validación en Fabric.

# NB_01 — Clasificación de textos con `ai.classify`
**Serie:** Microsoft Fabric AI Functions con Pandas  
**Modelo:** `gpt-5-mini`  
**Entorno:** Notebook Python de Microsoft Fabric

## Objetivo
Clasificar solicitudes de soporte con las mismas diez entradas y seis categorías de la versión PySpark, medir resultados, tiempo, memoria y consumo del modelo integrado.

## 1. Requisitos
Capacidad pagada F2 o superior (o P), configuración de tenant para Azure OpenAI habilitada y permisos de ejecución. Según la región puede requerirse procesamiento entre geografías. Usar datos ficticios. Consultar [requisitos e instalación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview). En runtime Python se requieren los wheels de SynapseML; en Runtime 2.0 Spark seguir el procedimiento temporal específico de Microsoft. No se necesita Spark para este laboratorio.

In [1]:
# Instalación opcional en runtime Python; ejecutar una vez si faltan las bibliotecas.
INSTALL = False
if INSTALL:
    import subprocess, sys
    from urllib.request import urlretrieve
    wheels = []
    for url, name in [
        ("https://aka.ms/fabric-aifunctions-whl", "synapseml_internal-latest-py3-none-any.whl"),
        ("https://aka.ms/fabric-synapseml-core-whl", "synapseml_core-latest-py3-none-any.whl")
    ]:
        urlretrieve(url, name)
        wheels.append(name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *wheels])
# Reiniciar la sesión si el runtime lo requiere.

StatementMeta(, 21c842a7-23b5-44d9-98f2-a3e0e9cb36a3, 3, Finished, Available, Finished, False)

## 2. Configuración del modelo
Usaremos concurrencia 10 para este laboratorio pequeño (el valor predeterminado documentado es 200). La barra `stats` muestra métricas y proyecciones durante la ejecución. [Configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration).

In [2]:
import pandas as pd
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from time import perf_counter
import uuid
MODEL_NAME = "gpt-5-mini"
CONCURRENCY = 10
aifunc.default_conf.model_deployment_name = MODEL_NAME
aifunc.default_conf.reasoning_effort = "low"
aifunc.default_conf.concurrency = CONCURRENCY
aifunc.default_conf.progress_bar_mode = "stats"

StatementMeta(, 21c842a7-23b5-44d9-98f2-a3e0e9cb36a3, 4, Finished, Available, Finished, False)

## 3. Referencia oficial del método
[Microsoft Learn — Use ai.classify with pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/classify).

Ejemplo de sintaxis de Microsoft (referencial):
```python
df["classification"] = df["input"].ai.classify("category1", "category2", "category3")
```
| Parámetro | Tipo | Requisito / valor predeterminado |
|---|---|---|
| `labels` | strings posicionales | Obligatorio, mínimo uno; se recomiendan dos o más |
| `conf` | Conf | Opcional, configuración global si se omite |
| `column_type` | string | Opcional; usar `path` para rutas de archivos como texto |

Retorna una Series de Pandas, con una etiqueta por entrada o un valor nulo si no se logra clasificar. No recibe `input_col`, `output_col` ni `error_col` como en PySpark.

## 4. Crear un conjunto de datos de prueba

Utilizaremos solicitudes ficticias de soporte tecnológico. Cada registro contiene una descripción que deberá clasificarse en una de estas categorías:

- `Infraestructura`
- `Bases de datos`
- `Seguridad`
- `Redes`
- `Aplicaciones`
- `Otros`

Las categorías deben ser suficientemente claras y cubrir los escenarios esperados. La calidad de la clasificación depende, entre otros factores, de la claridad del texto y de las etiquetas suministradas.

In [3]:
data = [
    (1, "El servidor virtual presenta alta utilización de memoria y CPU."),
    (2, "La consulta SQL demora varios minutos y bloquea otras transacciones."),
    (3, "Se detectaron intentos de acceso no autorizado a una cuenta."),
    (4, "Los usuarios pierden conectividad con la red inalámbrica de la oficina."),
    (5, "La aplicación de gestión muestra un error al guardar una solicitud."),
    (6, "Se requiere ampliar el almacenamiento del clúster de procesamiento."),
    (7, "Una tabla presenta registros duplicados después de la carga ETL."),
    (8, "El certificado TLS del portal web está próximo a vencer."),
    (9, "El servicio de DNS no resuelve el nombre del servidor interno."),
    (10, "Se solicita incorporar un nuevo campo al formulario de atención.")
]

df = pd.DataFrame(data, columns=["id_solicitud", "descripcion"])

display(df)

StatementMeta(, 21c842a7-23b5-44d9-98f2-a3e0e9cb36a3, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bc62872b-c156-47ac-9cb6-60f0353fc0dd)

## 5. Ejecutar la clasificación
El operador `*` expande las etiquetas. Conservamos la Series original para no perder los metadatos de estadísticas. Cada nueva llamada puede generar consumo.

In [4]:
LABELS = ["Infraestructura", "Bases de datos", "Seguridad", "Redes", "Aplicaciones", "Otros"]
run_id = str(uuid.uuid4())
started_utc = datetime.now(timezone.utc).isoformat()
input_bytes = int(df.memory_usage(deep=True).sum())
t0 = perf_counter()
clasificaciones = df["descripcion"].ai.classify(*LABELS)
elapsed_seconds = perf_counter() - t0
stats_raw = clasificaciones.ai.stats
resultado = df.assign(categoria=clasificaciones)
output_bytes = int(resultado.memory_usage(deep=True).sum())
display(resultado)

StatementMeta(, 21c842a7-23b5-44d9-98f2-a3e0e9cb36a3, 6, Finished, Available, Finished, False)

ai.classify: 100%|██████████| 10/10 [00:01<00:00,  5.91it/s, cached=0, in=754, out=774, CU-s=52.67]                            


SynapseWidget(Synapse.DataFrame, 42058f8a-840e-4a5e-bf66-7f776f419ba4)

## 6. Validación de resultados
Identificar nulos y valores fuera del catálogo. Un nulo no permite determinar por sí solo la causa. Revisar las categorías con criterios de negocio.

In [7]:
# Mostrar distribución de categorías, si existe la columna
if "categoria" in resultado.columns:
    display(resultado["categoria"].value_counts(dropna=False))
else:
    print("La columna 'categoria' no existe en 'resultado'.")

# Detectar valores fuera del catálogo solo si hay filas y la columna existe
if "categoria" in resultado.columns and len(resultado) > 0:
    invalidos = resultado[~resultado["categoria"].isin(LABELS)]
    # Si el DataFrame está vacío, evitar que display intente inferir tipos sobre una estructura especial vacía
    if len(invalidos) > 0:
        display(invalidos)
    else:
        print("No se encontraron valores fuera del catálogo de LABELS.")
else:
    print("No se pudo evaluar 'invalidos' porque 'resultado' está vacío o sin columna 'categoria'.")

print(f"Tiempo: {elapsed_seconds:.3f} s; entrada: {input_bytes} bytes; salida: {output_bytes} bytes")

StatementMeta(, 21c842a7-23b5-44d9-98f2-a3e0e9cb36a3, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c5b9955-b4a2-4ef2-9f45-e462144e62fb)

No se encontraron valores fuera del catálogo de LABELS.
Tiempo: 6.469 s; entrada: 2060 bytes; salida: 2140 bytes


## 7. Estadísticas y tokens
[Facturación y estadísticas oficiales](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing). Las métricas incluyen filas exitosas, excepciones, filas no evaluadas, filtros de seguridad y tokens de entrada, caché, salida y razonamiento. No sustituir métricas ausentes por cero.

In [8]:
display(stats_raw)
if isinstance(stats_raw, pd.DataFrame) and len(stats_raw) == 1:
    stats = stats_raw.iloc[0].to_dict()
elif isinstance(stats_raw, pd.Series):
    stats = stats_raw.to_dict()
elif isinstance(stats_raw, dict):
    stats = stats_raw
else:
    raise RuntimeError("Revisar estructura de ai.stats antes de estimar consumo")
def metric(key):
    v = stats.get(key)
    return None if v is None or pd.isna(v) else int(v)
I, C, O, R = [metric(k) for k in ("input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens")]
print(dict(input_tokens=I, cached_tokens=C, output_tokens=O, reasoning_tokens=R))

StatementMeta(, 21c842a7-23b5-44d9-98f2-a3e0e9cb36a3, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6ae386e3-a7a3-4147-828a-16f5b5a51e8c)

{'input_tokens': 754, 'cached_tokens': 0, 'output_tokens': 774, 'reasoning_tokens': 256}


## 8. Estimación de CU-segundos
Tarifas de referencia verificadas en Microsoft Learn: entrada 8,40; caché 0,84; salida 67,23 CU-s por mil tokens.
$$CU_s = \frac{(I-C)\times 8.40+C\times 0.84+O\times 67.23}{1000}$$
Ejemplo ilustrativo: I=1000, C=200 y O=100 equivalen a 13,611 CU-s. El razonamiento no se suma nuevamente si ya está incluido en salida. La estimación solo corresponde al endpoint integrado; no incluye cómputo del notebook.

In [9]:
estimated_cu_seconds = None
if (stats.get("model") in (MODEL_NAME, "gpt-5-mini-2025-08-07")
    and stats.get("client_type") == "fabric_llm_endpoint"
    and all(v is not None for v in (I,C,O)) and 0 <= C <= I and O >= 0):
    estimated_cu_seconds = ((I-C)*8.40+C*0.84+O*67.23)/1000
print("CU-segundos estimados:", estimated_cu_seconds)

StatementMeta(, 21c842a7-23b5-44d9-98f2-a3e0e9cb36a3, 11, Finished, Available, Finished, False)

CU-segundos estimados: 58.369620000000005


## 9. Auditoría
Se añaden tiempo, memoria y concurrencia al esquema de referencia de PySpark. La memoria medida corresponde a los DataFrames, no al pico de memoria del proceso.

In [10]:
audit = dict(run_id=run_id, execution_timestamp_utc=started_utc,
    notebook_name="NB_01_AI_CLASSIFY_PANDAS", operation="ai.classify",
    model_configured=MODEL_NAME, model_reported=stats.get("model"),
    num_input_rows=len(df), client_type=stats.get("client_type"),
    estimated_cu_seconds=estimated_cu_seconds, elapsed_seconds=elapsed_seconds,
    input_bytes=input_bytes, output_bytes=output_bytes, concurrency=CONCURRENCY)
for key in ("num_successful", "num_exceptions", "num_unevaluated", "num_harmful", "input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens"):
    audit[key] = metric(key)
audit["execution_status"] = "SUCCESS" if audit["num_successful"] == len(df) and invalidos.empty else "INCOMPLETE"
audit_df = pd.DataFrame([audit])
display(audit_df)

StatementMeta(, 21c842a7-23b5-44d9-98f2-a3e0e9cb36a3, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5b494767-4d74-4b6e-880d-3b0491bed795)

## 10. Persistencia opcional en Delta
Usar `deltalake` y un Lakehouse accesible. Se escribe en una ruta Delta independiente bajo Files, no se registra automáticamente como tabla SQL. Instalar deltalake si falta. Revisar permisos y esquema antes de habilitar.

In [ ]:
SAVE_DELTA = False
if SAVE_DELTA:
    from deltalake import write_deltalake
    import pyarrow as pa
    schema = pa.schema([(k, pa.int64() if k.startswith("num_") or k.endswith("_tokens") or k in ("input_bytes", "output_bytes", "concurrency") else pa.float64() if k in ("estimated_cu_seconds", "elapsed_seconds") else pa.string()) for k in audit_df.columns])
    write_deltalake("/lakehouse/default/Files/audit_ai_functions_pandas", pa.Table.from_pandas(audit_df, schema=schema, preserve_index=False), mode="append")

## 11. Comparación con PySpark y conclusiones
En Pandas se invoca sobre Series y se materializa el resultado en la llamada; PySpark usa transformaciones distribuidas. Comparar con NB_01 usando las mismas entradas, modelo y configuración comparable. Registrar diferencias de concurrencia, caché y capacidad; no afirmar superioridad sin benchmark.

## Referencias
- [Método classify](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/classify)
- [Configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration)
- [Descripción general](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)